# S6E8 V5: Stability-Aware Greedy Research Stack

## Predicting Smartphone Addiction with nested ensemble selection

This notebook extends the validated V4 pipeline with a memory-safe, stability-aware
ensemble selector. It keeps the independently trained XGBoost, CatBoost, histogram,
and Lookup Transformer members, reuses their V4 checkpoints, and evaluates every
aligned public OOF member as a candidate.

The V5 contribution is a nested Caruana-style ensemble selection procedure:

1. each meta-validation fold selects members using only the other folds;
2. repeated member selection learns discrete non-negative weights;
3. a candidate is accepted only when its mean fold AUC improves and its worst-fold
   loss remains bounded;
4. a compact missingness-aware logistic meta-model is trained on the selected set;
5. the final member weights are refitted on all OOF predictions only after the
   nested estimate has been produced.

This is a serious competition experiment, not a leaderboard promise. Every public
member remains credited in the exported provenance record.


## Research rules used in this notebook

- The identifier is never used as a predictive feature.
- Every target-derived feature is fitted inside the current training fold.
- All base models share exactly the same frozen outer folds.
- Greedy member selection is repeated inside each meta-validation split.
- Selection requires fold-level stability, not only a larger global OOF number.
- Public OOF members must provide aligned train and test predictions and provenance.
- Google Drive checkpoints make completed base-model folds reusable after a disconnect.
- The public leaderboard is used only after a submission is frozen.

For the final experiment, select a T4 GPU in Colab and leave `FULL_RUN = True`.
The V5 selector itself is CPU/RAM work and is designed for the standard Colab memory profile.


## 1. Install the competition dependencies

PyTorch is already available in the Colab GPU image. The following cell installs Kaggle access, CatBoost, and a recent CUDA-aware XGBoost build.

In [ ]:
!pip -q install "kaggle>=1.7" "xgboost>=2.1" "catboost>=1.2"

In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb

from catboost import CatBoostClassifier, CatBoostError, Pool
from scipy.special import expit
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

PIPELINE_VERSION = "v5.0.0"
BASE_MODEL_CACHE_VERSION = "v4.0.0"
COMPETITION = "playground-series-s6e8"
ORIGINAL_DATASET = "jayjoshi37/smartphone-usage-and-addiction-prediction"
TARGET = "addicted_label"
ID_COL = "id"

# Recommended final run. False is only a short pipeline test.
FULL_RUN = True
USE_GOOGLE_DRIVE = True
RUN_XGBOOST = True
RUN_CATBOOST = True
RUN_HISTOGRAM_NB = True
RUN_LOOKUP_TRANSFORMER = True
IMPORT_PUBLIC_OOF_LIBRARY = True
EXTRA_NEURAL_SEED = False
SAVE_FULL_MEMBER_MATRIX = False

# The fixed V4 stack remains as a comparable baseline.
MAX_STACK_MEMBERS = 24

# V5 greedy-selection controls. These values are precommitted and are not tuned
# against the public leaderboard.
GREEDY_MAX_STEPS = 12 if FULL_RUN else 5
GREEDY_SEARCH_ROWS = 100_000 if FULL_RUN else 30_000
GREEDY_TOP_GRADIENT = 16
GREEDY_TOP_INDIVIDUAL = 8
GREEDY_TOP_DIVERSE = 8
GREEDY_CONFIRM_TOP = 8
GREEDY_MIN_MEAN_GAIN = 0.000002
GREEDY_MAX_FOLD_LOSS = 0.000030
GREEDY_META_C = 0.03

FOLD_SEED = 42
N_SPLITS = 5 if FULL_RUN else 3
TE_INNER_SPLITS = 4 if FULL_RUN else 2
XGB_ESTIMATORS = 4500 if FULL_RUN else 700
CATBOOST_ITERATIONS = 3500 if FULL_RUN else 600
LOOKUP_EPOCHS = 14 if FULL_RUN else 4
LOOKUP_PATIENCE = 3 if FULL_RUN else 1
LOOKUP_BATCH_SIZE = 4096 if FULL_RUN else 2048
LOOKUP_SEEDS = [2718, 31415] if EXTRA_NEURAL_SEED else [2718]
NB_BIN_COUNTS = [16, 32, 64] if FULL_RUN else [24]

RAW_NUMERIC = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
]
RAW_CATEGORICAL = [
    "gender",
    "stress_level",
    "academic_work_impact",
]
RAW_COLUMNS = RAW_NUMERIC + RAW_CATEGORICAL

PAIR_KEYS = [
    ("daily_screen_time_hours", "social_media_hours"),
    ("daily_screen_time_hours", "gaming_hours"),
    ("daily_screen_time_hours", "work_study_hours"),
    ("daily_screen_time_hours", "weekend_screen_time"),
    ("notifications_per_day", "app_opens_per_day"),
]

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if GPU_AVAILABLE else "cpu")
GPU_DESCRIPTION = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "No NVIDIA GPU"

# Keep the exact V4 signature so completed base-model checkpoints are reused.
base_config_for_hash = {
    "pipeline": BASE_MODEL_CACHE_VERSION,
    "full_run": FULL_RUN,
    "fold_seed": FOLD_SEED,
    "n_splits": N_SPLITS,
    "te_inner_splits": TE_INNER_SPLITS,
    "xgb_estimators": XGB_ESTIMATORS,
    "catboost_iterations": CATBOOST_ITERATIONS,
    "lookup_epochs": LOOKUP_EPOCHS,
    "lookup_seeds": LOOKUP_SEEDS,
    "nb_bins": NB_BIN_COUNTS,
    "import_public_oof": IMPORT_PUBLIC_OOF_LIBRARY,
}
CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(base_config_for_hash, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

experiment_config_for_hash = {
    "pipeline": PIPELINE_VERSION,
    "base_signature": CONFIG_SIGNATURE,
    "greedy_max_steps": GREEDY_MAX_STEPS,
    "greedy_search_rows": GREEDY_SEARCH_ROWS,
    "greedy_min_mean_gain": GREEDY_MIN_MEAN_GAIN,
    "greedy_max_fold_loss": GREEDY_MAX_FOLD_LOSS,
    "greedy_meta_c": GREEDY_META_C,
}
EXPERIMENT_SIGNATURE = hashlib.sha256(
    json.dumps(experiment_config_for_hash, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

print("Run configuration")
print(f"  Pipeline:              {PIPELINE_VERSION}")
print(f"  Base cache version:     {BASE_MODEL_CACHE_VERSION}")
print(f"  Full run:              {FULL_RUN}")
print(f"  Frozen folds:          {N_SPLITS}, seed={FOLD_SEED}")
print(f"  GPU available:         {GPU_AVAILABLE}")
print(f"  GPU:                   {GPU_DESCRIPTION}")
print(f"  Lookup seeds:          {LOOKUP_SEEDS}")
print(f"  Base config signature: {CONFIG_SIGNATURE}")
print(f"  V5 experiment:         {EXPERIMENT_SIGNATURE}")

if RUN_LOOKUP_TRANSFORMER and not GPU_AVAILABLE:
    raise RuntimeError(
        "The Lookup Transformer requires a GPU for this dataset. "
        "In Colab, choose Runtime > Change runtime type > T4 GPU."
    )


## 2. Persistent storage and secure Kaggle authentication

Completed fold predictions are stored in Google Drive. If Colab disconnects, reconnect and run the notebook again: completed folds will be loaded instead of retrained.

Create a Colab Secret named KAGGLE_API_TOKEN, enable notebook access, and never paste the token directly into a code cell. If a token has appeared in a screenshot or public notebook, revoke it and generate a new one before continuing.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    # This root intentionally remains compatible with the completed V4 run.
    PERSIST_ROOT = Path("/content/drive/MyDrive/s6e8_top3_research_v4")
elif IN_KAGGLE:
    PERSIST_ROOT = Path("/kaggle/working/s6e8_top3_research_v4")
else:
    PERSIST_ROOT = Path("./s6e8_top3_research_v4")

PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints" / CONFIG_SIGNATURE
ARTIFACT_DIR = PERSIST_ROOT / "artifacts_v5" / EXPERIMENT_SIGNATURE
SUBMISSION_DIR = PERSIST_ROOT / "submissions_v5" / EXPERIMENT_SIGNATURE
ANCHOR_DIR = PERSIST_ROOT / "anchors"

for directory in [CHECKPOINT_DIR, ARTIFACT_DIR, SUBMISSION_DIR, ANCHOR_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

KAGGLE_TOKEN_READY = False

if IN_COLAB:
    from google.colab import userdata

    try:
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Create the KAGGLE_API_TOKEN Colab Secret and enable notebook access."
        ) from exc

    if not kaggle_token:
        raise RuntimeError("The KAGGLE_API_TOKEN secret is empty.")

    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    KAGGLE_TOKEN_READY = True
elif os.environ.get("KAGGLE_API_TOKEN"):
    KAGGLE_TOKEN_READY = True
elif IN_KAGGLE:
    print("Kaggle input mode detected; API authentication may not be required.")

print("Persistent root:", PERSIST_ROOT)
print("Reusing V4 checkpoints from:", CHECKPOINT_DIR)
print("V5 outputs will be written to:", SUBMISSION_DIR)
print("Kaggle authentication configured without printing the token.")


## 3. Download and validate the data

The public source dataset is used only as an external reference distribution. It is not blindly appended to the competition training data.

In [ ]:
if IN_KAGGLE and Path(f"/kaggle/input/{COMPETITION}").exists():
    COMP_DIR = Path(f"/kaggle/input/{COMPETITION}")
    DATA_ROOT = Path("/kaggle/working/s6e8_top3_data")
else:
    DATA_ROOT = Path("/content/s6e8_top3_data") if IN_COLAB else Path("./s6e8_top3_data")
    COMP_DIR = DATA_ROOT / "competition"
    COMP_DIR.mkdir(parents=True, exist_ok=True)

    required_competition_files = [
        COMP_DIR / "train.csv",
        COMP_DIR / "test.csv",
        COMP_DIR / "sample_submission.csv",
    ]
    if not all(path.exists() for path in required_competition_files):
        if not KAGGLE_TOKEN_READY:
            raise RuntimeError("Kaggle authentication is required to download the data.")
        subprocess.run(
            [
                "kaggle", "competitions", "download",
                "-c", COMPETITION,
                "-p", str(COMP_DIR),
            ],
            check=True,
        )
        for archive_path in COMP_DIR.glob("*.zip"):
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(COMP_DIR)

ORIG_DIR = DATA_ROOT / "original"
ORIG_DIR.mkdir(parents=True, exist_ok=True)
original_csvs = list(ORIG_DIR.rglob("*.csv"))

if not original_csvs:
    if not KAGGLE_TOKEN_READY:
        raise RuntimeError(
            "Attach the public source dataset or provide Kaggle API authentication."
        )
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", ORIGINAL_DATASET,
            "-p", str(ORIG_DIR),
            "--unzip",
        ],
        check=True,
    )
    original_csvs = list(ORIG_DIR.rglob("*.csv"))

required_paths = [
    COMP_DIR / "train.csv",
    COMP_DIR / "test.csv",
    COMP_DIR / "sample_submission.csv",
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing competition files: {missing_paths}")
if not original_csvs:
    raise FileNotFoundError("The original reference CSV was not found.")

ORIGINAL_CSV = original_csvs[0]

train = pd.read_csv(COMP_DIR / "train.csv")
test = pd.read_csv(COMP_DIR / "test.csv")
sample_submission = pd.read_csv(COMP_DIR / "sample_submission.csv")
original = pd.read_csv(ORIGINAL_CSV)

assert set(RAW_COLUMNS + [ID_COL, TARGET]).issubset(train.columns)
assert set(RAW_COLUMNS + [ID_COL]).issubset(test.columns)
assert set(RAW_COLUMNS + [TARGET]).issubset(original.columns)
assert list(sample_submission.columns) == [ID_COL, TARGET]
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert train[TARGET].isin([0, 1]).all()

y = train[TARGET].to_numpy(dtype="int8")

print(f"Train:       {train.shape}")
print(f"Test:        {test.shape}")
print(f"Original:    {original.shape}")
print(f"Target mean: {y.mean():.6f}")
display(train.head())

## 4. Freeze one outer cross-validation split

All model families use these exact folds. This is essential: stacking predictions from different fold definitions can create misleading OOF results.

In [ ]:
fold_ids = np.full(len(train), -1, dtype="int8")
outer_splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=FOLD_SEED,
)

for fold, (_, valid_idx) in enumerate(outer_splitter.split(train, y)):
    fold_ids[valid_idx] = fold

assert (fold_ids >= 0).all()

fold_table = pd.DataFrame({
    ID_COL: train[ID_COL],
    TARGET: y,
    "fold": fold_ids,
})
fold_table.to_csv(ARTIFACT_DIR / "fold_assignments.csv", index=False)

fold_summary = fold_table.groupby("fold")[TARGET].agg(["size", "mean"])
display(fold_summary)

## 5. Structural features with a behavioral interpretation

The synthetic generator contains useful constraints. Daily screen time can be decomposed into social media, gaming, work or study, and unallocated screen time. Weekend usage also follows a strong label-free relationship with daily usage. The regression below is fitted on train and test features without using the target.

In [ ]:
combined_daily = pd.concat(
    [train["daily_screen_time_hours"], test["daily_screen_time_hours"]],
    ignore_index=True,
).to_numpy(dtype="float64")
combined_weekend = pd.concat(
    [train["weekend_screen_time"], test["weekend_screen_time"]],
    ignore_index=True,
).to_numpy(dtype="float64")
weekend_valid = np.isfinite(combined_daily) & np.isfinite(combined_weekend)
WEEKEND_SLOPE, WEEKEND_INTERCEPT = np.polyfit(
    combined_daily[weekend_valid],
    combined_weekend[weekend_valid],
    deg=1,
)

def safe_ratio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce").replace(0, np.nan)
    return numerator / denominator

def add_structural_features(frame):
    result = frame[RAW_COLUMNS].copy()

    daily = pd.to_numeric(frame["daily_screen_time_hours"], errors="coerce")
    social = pd.to_numeric(frame["social_media_hours"], errors="coerce")
    gaming = pd.to_numeric(frame["gaming_hours"], errors="coerce")
    work = pd.to_numeric(frame["work_study_hours"], errors="coerce")
    sleep = pd.to_numeric(frame["sleep_hours"], errors="coerce")
    weekend = pd.to_numeric(frame["weekend_screen_time"], errors="coerce")
    notifications = pd.to_numeric(frame["notifications_per_day"], errors="coerce")
    opens = pd.to_numeric(frame["app_opens_per_day"], errors="coerce")

    result["allocated_screen"] = social + gaming + work
    result["other_screen"] = daily - result["allocated_screen"]
    result["weekend_delta"] = weekend - daily
    result["weekend_residual"] = weekend - (
        WEEKEND_INTERCEPT + WEEKEND_SLOPE * daily
    )
    result["sleep_deficit"] = 8.0 - sleep
    result["awake_hours"] = 24.0 - sleep
    result["social_share_of_screen"] = safe_ratio(social, daily)
    result["gaming_share_of_screen"] = safe_ratio(gaming, daily)
    result["work_share_of_screen"] = safe_ratio(work, daily)
    result["other_share_of_screen"] = safe_ratio(result["other_screen"], daily)
    result["screen_to_sleep_ratio"] = safe_ratio(daily, sleep)
    result["screen_share_of_awake"] = safe_ratio(daily, result["awake_hours"])
    result["notifications_per_open"] = safe_ratio(notifications, opens)
    result["opens_per_screen_hour"] = safe_ratio(opens, daily)
    result["digital_interruptions"] = notifications + opens
    result["engagement_product"] = social * opens
    result["missing_count"] = frame[RAW_COLUMNS].isna().sum(axis=1).astype("int8")
    result["complete_row"] = (result["missing_count"] == 0).astype("int8")
    result["many_missing"] = (result["missing_count"] >= 4).astype("int8")

    return result

train_features = add_structural_features(train)
test_features = add_structural_features(test)

print(f"Weekend relationship: weekend = {WEEKEND_INTERCEPT:.4f} + "
      f"{WEEKEND_SLOPE:.4f} * daily")
print("Structural feature count:", train_features.shape[1])

## 6. External reference-distribution features

Exact overlaps with the competition training rows are removed first. The remaining source rows provide empirical CDF positions, class-conditional CDF gaps, median distances, and coarse target-rate curves. No competition validation labels are used here.

In [ ]:
original = original.dropna(subset=[TARGET]).copy()
original[TARGET] = original[TARGET].astype("int8")

def normalized_row_hash(frame):
    normalized = pd.DataFrame(index=frame.index)
    for column in RAW_NUMERIC:
        normalized[column] = (
            pd.to_numeric(frame[column], errors="coerce")
            .round(8)
            .fillna(-999999.0)
        )
    for column in RAW_CATEGORICAL:
        normalized[column] = frame[column].astype("string").fillna("__MISSING__")
    return pd.util.hash_pandas_object(normalized[RAW_COLUMNS], index=False)

train_hashes = set(normalized_row_hash(train).to_numpy())
original_hashes = normalized_row_hash(original)
original = original.loc[~original_hashes.isin(train_hashes)].copy()
original = original.drop_duplicates(subset=RAW_COLUMNS).reset_index(drop=True)

ORIG_CDF_COLUMNS = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours",
]
ORIG_CLASS_COLUMNS = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours",
    "notifications_per_day",
    "app_opens_per_day",
]
ORIG_MEAN_COLUMNS = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "notifications_per_day",
    "app_opens_per_day",
]

def finite_values(series):
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype="float64")
    return values[np.isfinite(values)]

def empirical_cdf(values, sorted_reference):
    values = np.asarray(values, dtype="float64")
    result = np.full(len(values), np.nan, dtype="float64")
    valid = np.isfinite(values)
    result[valid] = (
        np.searchsorted(sorted_reference, values[valid], side="right")
        / max(len(sorted_reference), 1)
    )
    return result

def quantile_edges(values, bins=20):
    values = values[np.isfinite(values)]
    edges = np.unique(np.quantile(values, np.linspace(0, 1, bins + 1)))
    if len(edges) < 2:
        return np.array([-np.inf, np.inf])
    edges[0], edges[-1] = -np.inf, np.inf
    return edges

orig_y = original[TARGET].to_numpy(dtype="int8")
orig_global_mean = float(orig_y.mean())
orig_references = {"cdf": {}, "class_cdf": {}, "median": {}, "mean": {}}

for column in ORIG_CDF_COLUMNS:
    orig_references["cdf"][column] = np.sort(finite_values(original[column]))

for column in ORIG_CLASS_COLUMNS:
    values = pd.to_numeric(original[column], errors="coerce").to_numpy(dtype="float64")
    orig_references["class_cdf"][column] = {
        0: np.sort(values[(orig_y == 0) & np.isfinite(values)]),
        1: np.sort(values[(orig_y == 1) & np.isfinite(values)]),
    }
    orig_references["median"][column] = {
        "all": float(np.nanmedian(values)),
        0: float(np.nanmedian(values[orig_y == 0])),
        1: float(np.nanmedian(values[orig_y == 1])),
    }

for column in ORIG_MEAN_COLUMNS:
    values = pd.to_numeric(original[column], errors="coerce").to_numpy(dtype="float64")
    edges = quantile_edges(values, bins=20)
    bin_ids = np.searchsorted(edges, values, side="right") - 1
    bin_ids = np.clip(bin_ids, 0, len(edges) - 2)
    means = np.full(len(edges) - 1, orig_global_mean, dtype="float64")
    for bin_id in range(len(means)):
        mask = (bin_ids == bin_id) & np.isfinite(values)
        if mask.any():
            means[bin_id] = float(orig_y[mask].mean())
    orig_references["mean"][column] = {"edges": edges, "means": means}

def add_original_reference_features(features, raw_frame):
    result = features.copy()

    for column in ORIG_CDF_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        result[f"{column}__orig_cdf"] = empirical_cdf(
            values, orig_references["cdf"][column]
        )

    for column in ORIG_CLASS_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        cdf_0 = empirical_cdf(values, orig_references["class_cdf"][column][0])
        cdf_1 = empirical_cdf(values, orig_references["class_cdf"][column][1])
        result[f"{column}__orig_cdf_gap"] = cdf_0 - cdf_1

        medians = orig_references["median"][column]
        result[f"{column}__orig_median_distance"] = np.abs(values - medians["all"])
        result[f"{column}__orig_y0_median_distance"] = np.abs(values - medians[0])
        result[f"{column}__orig_y1_median_distance"] = np.abs(values - medians[1])

    for column in ORIG_MEAN_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        reference = orig_references["mean"][column]
        bin_ids = np.searchsorted(reference["edges"], values, side="right") - 1
        valid = np.isfinite(values)
        bin_ids = np.clip(bin_ids, 0, len(reference["means"]) - 1)
        encoded = np.full(len(values), orig_global_mean, dtype="float64")
        encoded[valid] = reference["means"][bin_ids[valid]]
        result[f"{column}__orig_target_mean"] = encoded

    return result

train_features = add_original_reference_features(train_features, train)
test_features = add_original_reference_features(test_features, test)

print(f"Clean original reference rows: {len(original):,}")
print("Feature count after external statistics:", train_features.shape[1])

## 7. Exact-value codes, pair keys, and frequency encoding

The competition data repeats many exact numerical values. We preserve those identities as integer keys. Target-free frequency encoding may use the combined train and test feature distribution. Pair keys are deliberately limited to five interpretable relationships to control sparsity.

In [ ]:
def canonical_key(series, numeric):
    if numeric:
        return (
            pd.to_numeric(series, errors="coerce")
            .round(8)
            .astype("string")
            .fillna("__MISSING__")
        )
    return series.astype("string").fillna("__MISSING__")

exact_feature_names = []
train_code_columns = []
test_code_columns = []
exact_cardinalities = []

for column in RAW_COLUMNS:
    combined_key = pd.concat(
        [
            canonical_key(train[column], column in RAW_NUMERIC),
            canonical_key(test[column], column in RAW_NUMERIC),
        ],
        ignore_index=True,
    )
    codes, uniques = pd.factorize(combined_key, sort=True)
    codes = codes.astype("int32") + 1

    exact_feature_names.append(column)
    train_code_columns.append(codes[: len(train)])
    test_code_columns.append(codes[len(train):])
    exact_cardinalities.append(len(uniques) + 1)

raw_train_codes = np.column_stack(train_code_columns).astype("int32")
raw_test_codes = np.column_stack(test_code_columns).astype("int32")

raw_position = {name: index for index, name in enumerate(RAW_COLUMNS)}

for left, right in PAIR_KEYS:
    left_index = raw_position[left]
    right_index = raw_position[right]
    combined_pairs = pd.DataFrame({
        "left": np.concatenate([
            raw_train_codes[:, left_index], raw_test_codes[:, left_index]
        ]),
        "right": np.concatenate([
            raw_train_codes[:, right_index], raw_test_codes[:, right_index]
        ]),
    })
    pair_hash = pd.util.hash_pandas_object(combined_pairs, index=False)
    pair_codes, pair_uniques = pd.factorize(pair_hash, sort=False)
    pair_codes = pair_codes.astype("int32") + 1

    pair_name = f"{left}__PAIR__{right}"
    exact_feature_names.append(pair_name)
    train_code_columns.append(pair_codes[: len(train)])
    test_code_columns.append(pair_codes[len(train):])
    exact_cardinalities.append(len(pair_uniques) + 1)

exact_train_codes = np.column_stack(train_code_columns).astype("int32")
exact_test_codes = np.column_stack(test_code_columns).astype("int32")
TE_ALPHAS = np.array(
    [20.0] * len(RAW_COLUMNS) + [50.0] * len(PAIR_KEYS),
    dtype="float32",
)

for feature_index, feature_name in enumerate(exact_feature_names):
    combined_codes = np.concatenate([
        exact_train_codes[:, feature_index],
        exact_test_codes[:, feature_index],
    ])
    counts = np.bincount(
        combined_codes,
        minlength=exact_cardinalities[feature_index],
    )
    train_features[f"{feature_name}__freq"] = (
        counts[exact_train_codes[:, feature_index]] / len(combined_codes)
    ).astype("float32")
    test_features[f"{feature_name}__freq"] = (
        counts[exact_test_codes[:, feature_index]] / len(combined_codes)
    ).astype("float32")

# Preserve only the three genuine categorical columns as integer tree inputs.
tree_train_features = train_features.copy()
tree_test_features = test_features.copy()
for column in RAW_CATEGORICAL:
    index = raw_position[column]
    tree_train_features[column] = raw_train_codes[:, index]
    tree_test_features[column] = raw_test_codes[:, index]

X_tree = tree_train_features.astype("float32").to_numpy()
X_test_tree = tree_test_features.astype("float32").to_numpy()
X_tree[~np.isfinite(X_tree)] = np.nan
X_test_tree[~np.isfinite(X_test_tree)] = np.nan
TREE_FEATURE_NAMES = list(tree_train_features.columns)

print("Tree matrix:", X_tree.shape)
print("Exact TE matrix:", exact_train_codes.shape)
print("Largest exact cardinalities:")
display(
    pd.Series(exact_cardinalities, index=exact_feature_names)
    .sort_values(ascending=False)
    .head(10)
    .to_frame("cardinality")
)

## 8. Checkpoint and member registry

Each member must provide one prediction for every training row and every test row. Fold caches are written atomically. This prevents a partial file from being mistaken for a completed fold.

In [ ]:
members = {}
training_records = []

def percentile_rank(values):
    values = np.asarray(values, dtype="float64")
    return (rankdata(values, method="average") - 0.5) / len(values)

def validate_predictions(oof, test_prediction, name):
    oof = np.asarray(oof, dtype="float64")
    test_prediction = np.asarray(test_prediction, dtype="float64")
    assert oof.shape == (len(train),), f"Invalid OOF shape for {name}"
    assert test_prediction.shape == (len(test),), f"Invalid test shape for {name}"
    assert np.isfinite(oof).all(), f"Non-finite OOF values for {name}"
    assert np.isfinite(test_prediction).all(), f"Non-finite test values for {name}"
    assert np.unique(oof).size > 100, f"Nearly constant OOF member: {name}"

def member_cache_path(name):
    return CHECKPOINT_DIR / f"member__{name}.npz"

def fold_cache_path(name, fold):
    return CHECKPOINT_DIR / f"fold_{fold}__{name}.npz"

def save_npz_atomic(path, **arrays):
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def register_member(name, oof, test_prediction, family, persist=True):
    validate_predictions(oof, test_prediction, name)
    members[name] = {
        "oof": np.asarray(oof, dtype="float64"),
        "test": np.asarray(test_prediction, dtype="float64"),
        "family": family,
    }
    if persist:
        save_npz_atomic(
            member_cache_path(name),
            oof=members[name]["oof"],
            test=members[name]["test"],
        )
    print(f"Registered {name}: OOF AUC={roc_auc_score(y, oof):.6f}")

def try_load_member(name, family):
    path = member_cache_path(name)
    if not path.exists():
        return False
    cached = np.load(path)
    register_member(name, cached["oof"], cached["test"], family, persist=False)
    print(f"Loaded completed member from {path.name}")
    return True

def save_fold_prediction(name, fold, valid_idx, valid_prediction, test_prediction):
    save_npz_atomic(
        fold_cache_path(name, fold),
        valid_idx=np.asarray(valid_idx, dtype="int64"),
        valid_prediction=np.asarray(valid_prediction, dtype="float64"),
        test_prediction=np.asarray(test_prediction, dtype="float64"),
    )

def load_fold_prediction(name, fold, expected_valid_idx):
    path = fold_cache_path(name, fold)
    if not path.exists():
        return None
    cached = np.load(path)
    if not np.array_equal(cached["valid_idx"], expected_valid_idx):
        raise RuntimeError(f"Cached validation indices do not match for {name}, fold {fold}")
    if cached["test_prediction"].shape != (len(test),):
        raise RuntimeError(f"Cached test shape does not match for {name}, fold {fold}")
    return cached["valid_prediction"], cached["test_prediction"]

## 9. Fold-safe exact-value target encoding

The outer validation labels are never used to create their encodings. Inside each outer training partition, another stratified split creates cross-fitted training encodings. Validation and test encodings are fitted only on the complete outer training partition.

In [ ]:
def fit_smoothed_table(codes, labels, cardinality, alpha):
    prior = float(np.mean(labels))
    counts = np.bincount(codes, minlength=cardinality).astype("float64")
    positive_sums = np.bincount(
        codes,
        weights=labels,
        minlength=cardinality,
    ).astype("float64")
    table = (positive_sums + alpha * prior) / (counts + alpha)
    return table, prior

def apply_smoothed_table(codes, table, prior):
    result = np.full(len(codes), prior, dtype="float32")
    valid = (codes >= 0) & (codes < len(table))
    result[valid] = table[codes[valid]].astype("float32")
    return result

def build_outer_te_matrices(train_idx, valid_idx, fold):
    encoded_train = np.zeros(
        (len(train_idx), exact_train_codes.shape[1]),
        dtype="float32",
    )
    encoded_valid = np.zeros(
        (len(valid_idx), exact_train_codes.shape[1]),
        dtype="float32",
    )
    encoded_test = np.zeros(
        (len(test), exact_train_codes.shape[1]),
        dtype="float32",
    )

    inner_splitter = StratifiedKFold(
        n_splits=TE_INNER_SPLITS,
        shuffle=True,
        random_state=FOLD_SEED * 100 + fold,
    )

    for inner_train_pos, inner_valid_pos in inner_splitter.split(
        train_idx, y[train_idx]
    ):
        fit_indices = train_idx[inner_train_pos]
        apply_indices = train_idx[inner_valid_pos]

        for column_index, cardinality in enumerate(exact_cardinalities):
            table, prior = fit_smoothed_table(
                exact_train_codes[fit_indices, column_index],
                y[fit_indices],
                cardinality,
                float(TE_ALPHAS[column_index]),
            )
            encoded_train[inner_valid_pos, column_index] = apply_smoothed_table(
                exact_train_codes[apply_indices, column_index],
                table,
                prior,
            )

    for column_index, cardinality in enumerate(exact_cardinalities):
        table, prior = fit_smoothed_table(
            exact_train_codes[train_idx, column_index],
            y[train_idx],
            cardinality,
            float(TE_ALPHAS[column_index]),
        )
        encoded_valid[:, column_index] = apply_smoothed_table(
            exact_train_codes[valid_idx, column_index], table, prior
        )
        encoded_test[:, column_index] = apply_smoothed_table(
            exact_test_codes[:, column_index], table, prior
        )

    return encoded_train, encoded_valid, encoded_test

## 10. Multi-variant exact-TE XGBoost

The two models use the same features but different tree growth strategies and random seeds. Their target encodings are shared inside each fold, so the second member adds diversity without doubling preprocessing time.

In [ ]:
XGB_SPECS = [
    {
        "name": "xgb_exact_depth7_seed42",
        "seed": 42,
        "params": {
            "max_depth": 7,
            "min_child_weight": 5.0,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "grow_policy": "depthwise",
        },
    },
    {
        "name": "xgb_exact_lossguide_seed2026",
        "seed": 2026,
        "params": {
            "max_depth": 0,
            "max_leaves": 96,
            "min_child_weight": 10.0,
            "subsample": 0.88,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.05,
            "reg_lambda": 3.0,
            "grow_policy": "lossguide",
        },
    },
]

if RUN_XGBOOST:
    pending_specs = []
    for spec in XGB_SPECS:
        if not try_load_member(spec["name"], "xgboost_exact_te"):
            pending_specs.append(spec)

    if pending_specs:
        xgb_oof = {
            spec["name"]: np.zeros(len(train), dtype="float64")
            for spec in pending_specs
        }
        xgb_test = {
            spec["name"]: np.zeros(len(test), dtype="float64")
            for spec in pending_specs
        }

        for fold in range(N_SPLITS):
            train_idx = np.where(fold_ids != fold)[0]
            valid_idx = np.where(fold_ids == fold)[0]
            print("\n" + "=" * 80)
            print(f"XGBoost outer fold {fold + 1}/{N_SPLITS}")

            fold_pending = []
            for spec in pending_specs:
                cached = load_fold_prediction(spec["name"], fold, valid_idx)
                if cached is None:
                    fold_pending.append(spec)
                else:
                    valid_prediction, test_prediction = cached
                    xgb_oof[spec["name"]][valid_idx] = valid_prediction
                    xgb_test[spec["name"]] += test_prediction / N_SPLITS
                    print(f"Loaded cached {spec['name']} fold {fold}")

            if not fold_pending:
                continue

            te_train, te_valid, te_test = build_outer_te_matrices(
                train_idx, valid_idx, fold
            )
            fold_train_matrix = np.column_stack([X_tree[train_idx], te_train])
            fold_valid_matrix = np.column_stack([X_tree[valid_idx], te_valid])
            fold_test_matrix = np.column_stack([X_test_tree, te_test])

            for spec in fold_pending:
                started = time.perf_counter()
                params = {
                    "objective": "binary:logistic",
                    "eval_metric": "auc",
                    "n_estimators": XGB_ESTIMATORS,
                    "learning_rate": 0.03,
                    "max_bin": 256,
                    "tree_method": "hist",
                    "device": "cuda" if GPU_AVAILABLE else "cpu",
                    "early_stopping_rounds": 200 if FULL_RUN else 60,
                    "n_jobs": -1,
                    "verbosity": 0,
                    "random_state": spec["seed"] + fold,
                    **spec["params"],
                }
                model = xgb.XGBClassifier(**params)

                try:
                    model.fit(
                        fold_train_matrix,
                        y[train_idx],
                        eval_set=[(fold_valid_matrix, y[valid_idx])],
                        verbose=250,
                    )
                except xgb.core.XGBoostError as exc:
                    if params["device"] != "cuda":
                        raise
                    print("CUDA XGBoost failed; retrying this member on CPU.")
                    print("Reason:", str(exc).splitlines()[0])
                    params["device"] = "cpu"
                    model = xgb.XGBClassifier(**params)
                    model.fit(
                        fold_train_matrix,
                        y[train_idx],
                        eval_set=[(fold_valid_matrix, y[valid_idx])],
                        verbose=250,
                    )

                best_iteration = getattr(model, "best_iteration", None)
                if best_iteration is None:
                    best_iteration = XGB_ESTIMATORS - 1
                iteration_range = (0, int(best_iteration) + 1)
                valid_prediction = model.predict_proba(
                    fold_valid_matrix, iteration_range=iteration_range
                )[:, 1]
                test_prediction = model.predict_proba(
                    fold_test_matrix, iteration_range=iteration_range
                )[:, 1]

                xgb_oof[spec["name"]][valid_idx] = valid_prediction
                xgb_test[spec["name"]] += test_prediction / N_SPLITS
                save_fold_prediction(
                    spec["name"], fold, valid_idx, valid_prediction, test_prediction
                )

                fold_auc = roc_auc_score(y[valid_idx], valid_prediction)
                minutes = (time.perf_counter() - started) / 60
                training_records.append({
                    "member": spec["name"],
                    "family": "xgboost_exact_te",
                    "fold": fold,
                    "auc": fold_auc,
                    "minutes": minutes,
                    "best_iteration": int(best_iteration),
                })
                print(
                    f"{spec['name']}: AUC={fold_auc:.6f}, "
                    f"best_iteration={best_iteration}, time={minutes:.1f} min"
                )

                del model, valid_prediction, test_prediction
                gc.collect()

            del te_train, te_valid, te_test
            del fold_train_matrix, fold_valid_matrix, fold_test_matrix
            gc.collect()

        for spec in pending_specs:
            register_member(
                spec["name"],
                xgb_oof[spec["name"]],
                xgb_test[spec["name"]],
                "xgboost_exact_te",
            )

## 11. CatBoost with native categoricals

CatBoost sees the raw ordered numeric values and only the three genuine categorical variables as categories. Treating every floating-point value as an unordered category can destroy useful order, so this member intentionally follows a different representation from XGBoost.

In [ ]:
CATBOOST_MEMBER = "catboost_native_seed314"

if RUN_CATBOOST and not try_load_member(CATBOOST_MEMBER, "catboost_native"):
    cat_train = train_features.copy()
    cat_test = test_features.copy()

    for column in RAW_CATEGORICAL:
        cat_train[column] = (
            cat_train[column].astype("string").fillna("__MISSING__").astype(str)
        )
        cat_test[column] = (
            cat_test[column].astype("string").fillna("__MISSING__").astype(str)
        )

    for column in cat_train.columns:
        if column not in RAW_CATEGORICAL:
            cat_train[column] = pd.to_numeric(
                cat_train[column], errors="coerce"
            ).replace([np.inf, -np.inf], np.nan)
            cat_test[column] = pd.to_numeric(
                cat_test[column], errors="coerce"
            ).replace([np.inf, -np.inf], np.nan)

    cat_oof = np.zeros(len(train), dtype="float64")
    cat_test_prediction = np.zeros(len(test), dtype="float64")

    for fold in range(N_SPLITS):
        train_idx = np.where(fold_ids != fold)[0]
        valid_idx = np.where(fold_ids == fold)[0]
        cached = load_fold_prediction(CATBOOST_MEMBER, fold, valid_idx)

        if cached is not None:
            valid_prediction, fold_test_prediction = cached
            cat_oof[valid_idx] = valid_prediction
            cat_test_prediction += fold_test_prediction / N_SPLITS
            print(f"Loaded cached {CATBOOST_MEMBER} fold {fold}")
            continue

        started = time.perf_counter()
        train_pool = Pool(
            cat_train.iloc[train_idx],
            y[train_idx],
            cat_features=RAW_CATEGORICAL,
        )
        valid_pool = Pool(
            cat_train.iloc[valid_idx],
            y[valid_idx],
            cat_features=RAW_CATEGORICAL,
        )
        test_pool = Pool(cat_test, cat_features=RAW_CATEGORICAL)

        common_params = {
            "iterations": CATBOOST_ITERATIONS,
            "learning_rate": 0.035,
            "depth": 8,
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "l2_leaf_reg": 5.0,
            "random_strength": 0.35,
            "bootstrap_type": "Bayesian",
            "bagging_temperature": 0.6,
            "border_count": 128,
            "random_seed": 314 + fold,
            "allow_writing_files": False,
            "verbose": 250,
        }

        try:
            model = CatBoostClassifier(
                **common_params,
                task_type="GPU" if GPU_AVAILABLE else "CPU",
                devices="0" if GPU_AVAILABLE else None,
            )
            model.fit(
                train_pool,
                eval_set=valid_pool,
                use_best_model=True,
                early_stopping_rounds=200 if FULL_RUN else 60,
            )
        except CatBoostError as exc:
            if not GPU_AVAILABLE:
                raise
            print("GPU CatBoost failed; retrying this fold on CPU.")
            print("Reason:", str(exc).splitlines()[0])
            model = CatBoostClassifier(
                **common_params,
                task_type="CPU",
                thread_count=-1,
            )
            model.fit(
                train_pool,
                eval_set=valid_pool,
                use_best_model=True,
                early_stopping_rounds=200 if FULL_RUN else 60,
            )

        valid_prediction = model.predict_proba(valid_pool)[:, 1]
        fold_test_prediction = model.predict_proba(test_pool)[:, 1]
        cat_oof[valid_idx] = valid_prediction
        cat_test_prediction += fold_test_prediction / N_SPLITS
        save_fold_prediction(
            CATBOOST_MEMBER,
            fold,
            valid_idx,
            valid_prediction,
            fold_test_prediction,
        )

        fold_auc = roc_auc_score(y[valid_idx], valid_prediction)
        minutes = (time.perf_counter() - started) / 60
        training_records.append({
            "member": CATBOOST_MEMBER,
            "family": "catboost_native",
            "fold": fold,
            "auc": fold_auc,
            "minutes": minutes,
            "best_iteration": int(model.get_best_iteration()),
        })
        print(f"CatBoost fold {fold}: AUC={fold_auc:.6f}, time={minutes:.1f} min")

        del model, train_pool, valid_pool, test_pool
        gc.collect()

    register_member(
        CATBOOST_MEMBER,
        cat_oof,
        cat_test_prediction,
        "catboost_native",
    )

## 12. Class-conditional histogram members

These models are intentionally different from boosted trees. Each feature contributes a smoothed class-conditional log-likelihood ratio. Their individual AUC may be lower, but a decorrelated error pattern can still improve a stack.

In [ ]:
NB_NUMERIC_FEATURES = RAW_NUMERIC + [
    "other_screen",
    "weekend_residual",
    "sleep_deficit",
    "social_share_of_screen",
    "gaming_share_of_screen",
    "work_share_of_screen",
    "screen_to_sleep_ratio",
    "notifications_per_open",
]
NB_DISCRETE_FEATURES = RAW_CATEGORICAL + [
    "missing_count",
    "complete_row",
    "many_missing",
]

nb_train_parts = []
nb_test_parts = []

for column in NB_NUMERIC_FEATURES:
    nb_train_parts.append(
        pd.to_numeric(train_features[column], errors="coerce").to_numpy(dtype="float64")
    )
    nb_test_parts.append(
        pd.to_numeric(test_features[column], errors="coerce").to_numpy(dtype="float64")
    )

for column in RAW_CATEGORICAL:
    column_index = raw_position[column]
    nb_train_parts.append(raw_train_codes[:, column_index].astype("float64"))
    nb_test_parts.append(raw_test_codes[:, column_index].astype("float64"))

for column in ["missing_count", "complete_row", "many_missing"]:
    nb_train_parts.append(train_features[column].to_numpy(dtype="float64"))
    nb_test_parts.append(test_features[column].to_numpy(dtype="float64"))

X_nb = np.column_stack(nb_train_parts)
X_test_nb = np.column_stack(nb_test_parts)
NB_DISCRETE_MASK = np.array(
    [False] * len(NB_NUMERIC_FEATURES) + [True] * len(NB_DISCRETE_FEATURES)
)

class HistogramNaiveBayes:
    def __init__(self, n_bins=32, alpha=1.0):
        self.n_bins = n_bins
        self.alpha = alpha
        self.specifications = []
        self.log_prior_odds = 0.0

    def _fit_transformer(self, values, discrete):
        finite = np.isfinite(values)
        if discrete:
            categories = np.unique(values[finite])
            return {"kind": "discrete", "categories": categories}

        if finite.sum() == 0:
            edges = np.array([], dtype="float64")
        else:
            quantiles = np.linspace(0, 1, self.n_bins + 1)[1:-1]
            edges = np.unique(np.quantile(values[finite], quantiles))
        return {"kind": "continuous", "edges": edges}

    @staticmethod
    def _transform(values, specification):
        finite = np.isfinite(values)
        if specification["kind"] == "continuous":
            edges = specification["edges"]
            result = np.searchsorted(edges, values, side="right").astype("int32")
            result[~finite] = len(edges) + 1
            return result, len(edges) + 2

        categories = specification["categories"]
        positions = np.searchsorted(categories, values)
        clipped = np.clip(positions, 0, max(len(categories) - 1, 0))
        known = finite & (positions < len(categories))
        if len(categories):
            known &= categories[clipped] == values
        result = np.full(len(values), len(categories), dtype="int32")
        result[known] = positions[known]
        return result, len(categories) + 1

    def fit(self, features, labels, discrete_mask):
        labels = np.asarray(labels, dtype="int8")
        positive_rate = np.clip(labels.mean(), 1e-6, 1 - 1e-6)
        self.log_prior_odds = float(np.log(positive_rate / (1 - positive_rate)))
        self.specifications = []

        for column_index in range(features.shape[1]):
            values = features[:, column_index]
            specification = self._fit_transformer(
                values, bool(discrete_mask[column_index])
            )
            bin_ids, cardinality = self._transform(values, specification)
            count_0 = np.bincount(
                bin_ids[labels == 0], minlength=cardinality
            ).astype("float64")
            count_1 = np.bincount(
                bin_ids[labels == 1], minlength=cardinality
            ).astype("float64")
            probability_0 = (count_0 + self.alpha) / (
                count_0.sum() + self.alpha * cardinality
            )
            probability_1 = (count_1 + self.alpha) / (
                count_1.sum() + self.alpha * cardinality
            )
            specification["log_ratio"] = np.log(probability_1) - np.log(probability_0)
            self.specifications.append(specification)

        return self

    def decision_function(self, features):
        score = np.full(len(features), self.log_prior_odds, dtype="float64")
        for column_index, specification in enumerate(self.specifications):
            bin_ids, _ = self._transform(features[:, column_index], specification)
            score += specification["log_ratio"][bin_ids]
        return score / math.sqrt(max(features.shape[1], 1))

    def predict_proba(self, features):
        return expit(np.clip(self.decision_function(features), -30, 30))

if RUN_HISTOGRAM_NB:
    for bin_count in NB_BIN_COUNTS:
        member_name = f"histogram_nb_{bin_count}bins"
        if try_load_member(member_name, "histogram_nb"):
            continue

        oof_prediction = np.zeros(len(train), dtype="float64")
        test_prediction = np.zeros(len(test), dtype="float64")

        for fold in range(N_SPLITS):
            train_idx = np.where(fold_ids != fold)[0]
            valid_idx = np.where(fold_ids == fold)[0]
            model = HistogramNaiveBayes(n_bins=bin_count, alpha=1.0)
            model.fit(X_nb[train_idx], y[train_idx], NB_DISCRETE_MASK)
            valid_prediction = model.predict_proba(X_nb[valid_idx])
            fold_test_prediction = model.predict_proba(X_test_nb)
            oof_prediction[valid_idx] = valid_prediction
            test_prediction += fold_test_prediction / N_SPLITS

        register_member(
            member_name,
            oof_prediction,
            test_prediction,
            "histogram_nb",
        )

## 13. Prepare compact exact-value vocabularies for the Lookup Transformer

Very rare identities are mapped to an unknown token. This caps memory while the periodic numerical branch still preserves the ordered value. During each fold, identities unseen in the outer training partition are also mapped to unknown.

In [ ]:
LOOKUP_MIN_COUNT = 2
LOOKUP_MAX_CARDINALITY = 50000

def compress_lookup_codes(train_codes, test_codes, min_count, max_cardinality):
    combined = np.concatenate([train_codes, test_codes])
    counts = np.bincount(combined)
    candidates = np.where(counts >= min_count)[0]
    candidates = candidates[candidates != 0]

    if len(candidates) > max_cardinality - 1:
        order = np.argsort(counts[candidates])[::-1]
        candidates = candidates[order[: max_cardinality - 1]]

    mapping = np.zeros(len(counts), dtype="int32")
    mapping[candidates] = np.arange(1, len(candidates) + 1, dtype="int32")
    return mapping[train_codes], mapping[test_codes], len(candidates) + 1

lookup_train_columns = []
lookup_test_columns = []
lookup_cardinalities = []

for column_index, column in enumerate(RAW_COLUMNS):
    compressed_train, compressed_test, cardinality = compress_lookup_codes(
        raw_train_codes[:, column_index],
        raw_test_codes[:, column_index],
        LOOKUP_MIN_COUNT,
        LOOKUP_MAX_CARDINALITY,
    )
    lookup_train_columns.append(compressed_train)
    lookup_test_columns.append(compressed_test)
    lookup_cardinalities.append(cardinality)

lookup_train_codes = np.column_stack(lookup_train_columns).astype("int32")
lookup_test_codes = np.column_stack(lookup_test_columns).astype("int32")
lookup_numeric_train = train[RAW_NUMERIC].apply(
    pd.to_numeric, errors="coerce"
).to_numpy(dtype="float32")
lookup_numeric_test = test[RAW_NUMERIC].apply(
    pd.to_numeric, errors="coerce"
).to_numpy(dtype="float32")

display(
    pd.Series(lookup_cardinalities, index=RAW_COLUMNS)
    .sort_values(ascending=False)
    .to_frame("lookup_cardinality")
)

## 14. Lookup Transformer architecture

Each feature receives an exact-value embedding and a learned column embedding. Numeric tokens additionally receive normalized values, missingness, and sine/cosine features at several frequencies. A small Transformer then learns interactions between the twelve feature tokens.

In [ ]:
class LookupTransformer(nn.Module):
    def __init__(
        self,
        cardinalities,
        n_numeric,
        d_model=48,
        n_heads=4,
        n_layers=2,
        dropout=0.10,
        mask_probability=0.03,
    ):
        super().__init__()
        self.n_features = len(cardinalities)
        self.n_numeric = n_numeric
        self.d_model = d_model
        self.mask_probability = mask_probability

        self.lookup_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, d_model, padding_idx=0)
            for cardinality in cardinalities
        ])
        self.column_embedding = nn.Parameter(
            torch.randn(self.n_features, d_model) * 0.02
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        frequencies = torch.tensor([1.0, 2.0, 4.0, 8.0], dtype=torch.float32)
        self.register_buffer("frequencies", frequencies)
        numeric_input_dim = 2 + 2 * len(frequencies)
        self.numeric_weight = nn.Parameter(
            torch.empty(n_numeric, numeric_input_dim, d_model)
        )
        self.numeric_bias = nn.Parameter(torch.zeros(n_numeric, d_model))
        nn.init.xavier_uniform_(self.numeric_weight)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 3,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
            norm=nn.LayerNorm(d_model),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, codes, numeric_values, numeric_missing):
        if self.training and self.mask_probability > 0:
            random_mask = torch.rand(codes.shape, device=codes.device) < self.mask_probability
            codes = codes.masked_fill(random_mask, 0)
            numeric_random_mask = random_mask[:, : self.n_numeric]
            numeric_values = numeric_values.masked_fill(numeric_random_mask, 0.0)
            numeric_missing = torch.maximum(
                numeric_missing,
                numeric_random_mask.to(numeric_missing.dtype),
            )

        lookup_tokens = torch.stack([
            embedding(codes[:, index])
            for index, embedding in enumerate(self.lookup_embeddings)
        ], dim=1)

        phase = (
            2.0
            * math.pi
            * numeric_values.unsqueeze(-1)
            * self.frequencies.view(1, 1, -1)
        )
        numeric_basis = torch.cat([
            numeric_values.unsqueeze(-1),
            numeric_missing.unsqueeze(-1),
            torch.sin(phase),
            torch.cos(phase),
        ], dim=-1)
        numeric_tokens = torch.einsum(
            "bnk,nkd->bnd", numeric_basis, self.numeric_weight
        ) + self.numeric_bias.unsqueeze(0)
        categorical_padding = torch.zeros(
            len(lookup_tokens),
            self.n_features - self.n_numeric,
            self.d_model,
            device=lookup_tokens.device,
            dtype=lookup_tokens.dtype,
        )
        lookup_tokens = lookup_tokens + torch.cat(
            [numeric_tokens, categorical_padding], dim=1
        )

        tokens = lookup_tokens + self.column_embedding.unsqueeze(0)
        cls = self.cls_token.expand(len(tokens), -1, -1)
        encoded = self.transformer(torch.cat([cls, tokens], dim=1))
        return self.head(encoded[:, 0]).squeeze(1)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def prepare_lookup_fold(train_idx, valid_idx):
    train_codes = lookup_train_codes[train_idx].copy()
    valid_codes = lookup_train_codes[valid_idx].copy()
    test_codes = lookup_test_codes.copy()

    for column_index, cardinality in enumerate(lookup_cardinalities):
        seen = np.zeros(cardinality, dtype=bool)
        seen[train_codes[:, column_index]] = True
        valid_column = valid_codes[:, column_index]
        test_column = test_codes[:, column_index]
        valid_codes[~seen[valid_column], column_index] = 0
        test_codes[~seen[test_column], column_index] = 0

    fit_numeric = lookup_numeric_train[train_idx]
    median = np.nanmedian(fit_numeric, axis=0)
    q1 = np.nanquantile(fit_numeric, 0.25, axis=0)
    q3 = np.nanquantile(fit_numeric, 0.75, axis=0)
    scale = np.maximum(q3 - q1, 1e-3)

    def normalize(values):
        missing = ~np.isfinite(values)
        normalized = (values - median) / scale
        normalized = np.clip(normalized, -8.0, 8.0)
        normalized[missing] = 0.0
        return normalized.astype("float32"), missing.astype("float32")

    train_numeric, train_missing = normalize(lookup_numeric_train[train_idx])
    valid_numeric, valid_missing = normalize(lookup_numeric_train[valid_idx])
    test_numeric, test_missing = normalize(lookup_numeric_test)

    return (
        train_codes,
        valid_codes,
        test_codes,
        train_numeric,
        valid_numeric,
        test_numeric,
        train_missing,
        valid_missing,
        test_missing,
    )

def make_lookup_loader(codes, numeric, missing, labels=None, training=False):
    tensors = [
        torch.from_numpy(codes.astype("int64", copy=False)),
        torch.from_numpy(numeric.astype("float32", copy=False)),
        torch.from_numpy(missing.astype("float32", copy=False)),
    ]
    if labels is not None:
        tensors.append(torch.from_numpy(np.asarray(labels, dtype="float32")))

    workers = 2 if IN_COLAB else 0
    return DataLoader(
        TensorDataset(*tensors),
        batch_size=LOOKUP_BATCH_SIZE if training else LOOKUP_BATCH_SIZE * 2,
        shuffle=training,
        num_workers=workers,
        pin_memory=GPU_AVAILABLE,
        persistent_workers=workers > 0,
        drop_last=False,
    )

@torch.no_grad()
def predict_lookup(model, loader):
    model.eval()
    predictions = []
    amp_enabled = DEVICE.type == "cuda"
    for batch in loader:
        codes, numeric, missing = [tensor.to(DEVICE, non_blocking=True) for tensor in batch[:3]]
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model(codes, numeric, missing)
        predictions.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(predictions)

def fit_lookup_model(train_loader, valid_loader, valid_labels, seed, state_path):
    seed_everything(seed)
    model = LookupTransformer(
        cardinalities=lookup_cardinalities,
        n_numeric=len(RAW_NUMERIC),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1.5e-3, weight_decay=2e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )
    loss_function = nn.BCEWithLogitsLoss()
    amp_enabled = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    best_auc = -np.inf
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, LOOKUP_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen_rows = 0

        for batch in train_loader:
            codes, numeric, missing, labels = [
                tensor.to(DEVICE, non_blocking=True) for tensor in batch
            ]
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits = model(codes, numeric, missing)
                loss = loss_function(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.detach()) * len(labels)
            seen_rows += len(labels)

        valid_prediction = predict_lookup(model, valid_loader)
        valid_auc = roc_auc_score(valid_labels, valid_prediction)
        scheduler.step(valid_auc)
        epoch_loss = running_loss / max(seen_rows, 1)
        learning_rate = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:02d}: loss={epoch_loss:.5f}, "
            f"valid_auc={valid_auc:.6f}, lr={learning_rate:.2e}"
        )

        if valid_auc > best_auc + 1e-6:
            best_auc = valid_auc
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            torch.save(
                {"state_dict": best_state, "auc": best_auc, "epoch": epoch},
                state_path,
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= LOOKUP_PATIENCE:
            print("Early stopping.")
            break

    if best_state is None:
        raise RuntimeError("Lookup Transformer did not produce a valid checkpoint.")
    model.load_state_dict(best_state)
    return model, best_auc

## 15. Train the Lookup Transformer on the frozen folds

The fold prediction cache is the restart boundary. A completed fold is never trained twice under the same configuration signature.

In [ ]:
if RUN_LOOKUP_TRANSFORMER:
    torch.set_float32_matmul_precision("high")

    for seed in LOOKUP_SEEDS:
        member_name = f"lookup_transformer_seed{seed}"
        if try_load_member(member_name, "lookup_transformer"):
            continue

        lookup_oof = np.zeros(len(train), dtype="float64")
        lookup_test_prediction = np.zeros(len(test), dtype="float64")

        for fold in range(N_SPLITS):
            train_idx = np.where(fold_ids != fold)[0]
            valid_idx = np.where(fold_ids == fold)[0]
            cached = load_fold_prediction(member_name, fold, valid_idx)

            if cached is not None:
                valid_prediction, fold_test_prediction = cached
                lookup_oof[valid_idx] = valid_prediction
                lookup_test_prediction += fold_test_prediction / N_SPLITS
                print(f"Loaded cached {member_name} fold {fold}")
                continue

            started = time.perf_counter()
            print("\n" + "=" * 80)
            print(f"{member_name}, fold {fold + 1}/{N_SPLITS}")

            prepared = prepare_lookup_fold(train_idx, valid_idx)
            (
                fold_train_codes,
                fold_valid_codes,
                fold_test_codes,
                fold_train_numeric,
                fold_valid_numeric,
                fold_test_numeric,
                fold_train_missing,
                fold_valid_missing,
                fold_test_missing,
            ) = prepared

            train_loader = make_lookup_loader(
                fold_train_codes,
                fold_train_numeric,
                fold_train_missing,
                y[train_idx],
                training=True,
            )
            valid_loader = make_lookup_loader(
                fold_valid_codes,
                fold_valid_numeric,
                fold_valid_missing,
                training=False,
            )
            test_loader = make_lookup_loader(
                fold_test_codes,
                fold_test_numeric,
                fold_test_missing,
                training=False,
            )

            state_path = CHECKPOINT_DIR / f"model__{member_name}__fold_{fold}.pt"
            model, best_auc = fit_lookup_model(
                train_loader,
                valid_loader,
                y[valid_idx],
                seed + fold,
                state_path,
            )
            valid_prediction = predict_lookup(model, valid_loader)
            fold_test_prediction = predict_lookup(model, test_loader)
            lookup_oof[valid_idx] = valid_prediction
            lookup_test_prediction += fold_test_prediction / N_SPLITS
            save_fold_prediction(
                member_name,
                fold,
                valid_idx,
                valid_prediction,
                fold_test_prediction,
            )

            minutes = (time.perf_counter() - started) / 60
            training_records.append({
                "member": member_name,
                "family": "lookup_transformer",
                "fold": fold,
                "auc": roc_auc_score(y[valid_idx], valid_prediction),
                "minutes": minutes,
                "best_iteration": None,
            })
            print(
                f"Lookup fold {fold}: AUC={best_auc:.6f}, "
                f"time={minutes:.1f} min"
            )

            del model, train_loader, valid_loader, test_loader, prepared
            del fold_train_codes, fold_valid_codes, fold_test_codes
            del fold_train_numeric, fold_valid_numeric, fold_test_numeric
            del fold_train_missing, fold_valid_missing, fold_test_missing
            gc.collect()
            torch.cuda.empty_cache()

        register_member(
            member_name,
            lookup_oof,
            lookup_test_prediction,
            "lookup_transformer",
        )

## 16. Optional licence-gated public OOF library

The strongest reproducible public stacks use a frozen five-fold OOF library. This section downloads three explicitly allowlisted public datasets used by the Apache-2.0 reproducible-stack research, removes members whose names indicate an unlicensed or level-2 prediction, and validates every remaining OOF/test pair before registration.

Public members are kept distinct from the original models trained above. Their source dataset and local file key are recorded in the provenance table. If a dataset layout changes, the importer prints diagnostics and continues with the original library instead of guessing.

In [ ]:
PUBLIC_OOF_DATASETS = [
    {
        "tag": "community",
        "slug": "szymonkapiski/s6e8-oof-library-47-models",
    },
    {
        "tag": "fm_lattice",
        "slug": "raykkretzschmar/s6e8-fm-lattice-blend-members",
    },
    {
        "tag": "golem",
        "slug": "dariushafshar/s6e8-golem-oof-library",
    },
]
PUBLIC_EXCLUDE_TOKENS = (
    "naji",
    "level2",
    "level_2",
    "meta",
    "stack",
    "submission",
)
MAX_PUBLIC_MEMBERS = 90
PUBLIC_LIBRARY_ROOT = PERSIST_ROOT / "public_oof_libraries"
PUBLIC_LIBRARY_ROOT.mkdir(parents=True, exist_ok=True)
public_provenance = []

def normalized_prediction_key(path, root, array_key=None):
    relative = str(path.relative_to(root).with_suffix("")).lower()
    text = relative if array_key is None else f"{relative}__{array_key.lower()}"
    text = re.sub(
        r"(validation|valid|oof|test|train|predictions|prediction|probs|prob)",
        "_",
        text,
    )
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

def collect_array_records(root):
    records = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".npy", ".npz"}:
            continue

        try:
            if path.suffix.lower() == ".npy":
                array = np.load(path, mmap_mode="r", allow_pickle=False)
                candidates = [(None, array.shape)]
            else:
                with np.load(path, allow_pickle=False) as archive:
                    candidates = [
                        (key, archive[key].shape)
                        for key in archive.files
                    ]
        except Exception as exc:
            print(f"Skipping unreadable public array {path.name}: {exc}")
            continue

        for array_key, shape in candidates:
            if not shape or len(shape) > 2:
                continue
            if shape[0] == len(train):
                role = "oof"
            elif shape[0] == len(test):
                role = "test"
            else:
                continue
            records.append({
                "path": path,
                "array_key": array_key,
                "shape": shape,
                "role": role,
                "key": normalized_prediction_key(path, root, array_key),
            })
    return records

def load_array_record(record):
    if record["path"].suffix.lower() == ".npy":
        array = np.load(record["path"], allow_pickle=False)
    else:
        with np.load(record["path"], allow_pickle=False) as archive:
            array = np.array(archive[record["array_key"]])
    if array.ndim == 1:
        array = array[:, None]
    return np.asarray(array)

def duplicate_fingerprint(prediction, sample_indices):
    return percentile_rank(np.asarray(prediction)[sample_indices])

def import_public_source(source):
    source_root = PUBLIC_LIBRARY_ROOT / source["tag"]
    source_root.mkdir(parents=True, exist_ok=True)

    if not any(source_root.rglob("*.npy")) and not any(source_root.rglob("*.npz")):
        if not KAGGLE_TOKEN_READY:
            print(f"Skipping {source['slug']}: Kaggle authentication is unavailable.")
            return 0
        print("Downloading public OOF dataset:", source["slug"])
        subprocess.run(
            [
                "kaggle", "datasets", "download",
                "-d", source["slug"],
                "-p", str(source_root),
                "--unzip",
            ],
            check=True,
        )

    records = collect_array_records(source_root)
    grouped = {}
    for record in records:
        grouped.setdefault(record["key"], {}).setdefault(record["role"], []).append(record)

    sample_size = min(30000, len(train))
    sample_indices = np.linspace(0, len(train) - 1, sample_size, dtype="int64")
    existing_fingerprints = [
        duplicate_fingerprint(member["oof"], sample_indices)
        for member in members.values()
    ]

    imported = 0
    for key, roles in sorted(grouped.items()):
        if imported >= MAX_PUBLIC_MEMBERS:
            break
        if "oof" not in roles or "test" not in roles:
            continue
        if any(token in key for token in PUBLIC_EXCLUDE_TOKENS):
            continue

        oof_record = roles["oof"][0]
        test_record = roles["test"][0]
        oof_matrix = load_array_record(oof_record)
        test_matrix = load_array_record(test_record)
        if oof_matrix.shape[1] != test_matrix.shape[1]:
            continue

        for column_index in range(oof_matrix.shape[1]):
            if imported >= MAX_PUBLIC_MEMBERS:
                break
            oof_prediction = np.asarray(
                oof_matrix[:, column_index], dtype="float64"
            )
            test_prediction = np.asarray(
                test_matrix[:, column_index], dtype="float64"
            )
            if not np.isfinite(oof_prediction).all() or not np.isfinite(test_prediction).all():
                continue
            if np.unique(oof_prediction).size < 100:
                continue

            # Convert every public stream to an AUC-safe common scale.
            oof_prediction = percentile_rank(oof_prediction)
            test_prediction = percentile_rank(test_prediction)
            member_auc = roc_auc_score(y, oof_prediction)
            if not 0.55 <= member_auc <= 0.985:
                continue

            fingerprint = duplicate_fingerprint(oof_prediction, sample_indices)
            is_duplicate = any(
                np.corrcoef(fingerprint, previous)[0, 1] > 0.999999
                for previous in existing_fingerprints
            )
            if is_duplicate:
                continue

            safe_key = re.sub(r"[^a-z0-9]+", "_", key)[:50]
            member_name = f"public_{source['tag']}_{safe_key}_{column_index}"
            register_member(
                member_name,
                oof_prediction,
                test_prediction,
                f"public_oof_{source['tag']}",
                persist=False,
            )
            existing_fingerprints.append(fingerprint)
            public_provenance.append({
                "member": member_name,
                "dataset": source["slug"],
                "key": key,
                "column": column_index,
                "oof_auc": member_auc,
                "oof_file": str(oof_record["path"]),
                "test_file": str(test_record["path"]),
            })
            imported += 1

    if imported == 0:
        print(f"No aligned OOF/test pairs imported from {source['slug']}.")
        print("Detected array files:")
        for path in list(sorted(source_root.rglob("*.npy")))[:15]:
            print(" ", path.relative_to(source_root))
    else:
        print(f"Imported {imported} validated members from {source['slug']}.")
    return imported

if IMPORT_PUBLIC_OOF_LIBRARY:
    if N_SPLITS != 5 or FOLD_SEED != 42:
        print("Public OOF import skipped: it requires the frozen 5-fold seed-42 protocol.")
    else:
        total_public_members = 0
        for source in PUBLIC_OOF_DATASETS:
            total_public_members += import_public_source(source)
        print(f"Total validated public members: {total_public_members}")

if public_provenance:
    pd.DataFrame(public_provenance).to_csv(
        ARTIFACT_DIR / "public_member_provenance.csv",
        index=False,
    )

## 17. Audit member strength and build the fixed V4 baseline

The original 24-member RAM-safe stack remains in this notebook as a controlled
baseline. V5 must beat it with nested OOF evidence before it becomes the primary
submission. The baseline selection itself is unchanged, so its result remains
comparable with the completed V4 run.


In [ ]:
if len(members) < 3:
    raise RuntimeError(
        "At least three completed members are required before stacking."
    )

all_member_names = list(members)
score_rows = []
for name in all_member_names:
    member_oof = members[name]["oof"]
    row = {
        "member": name,
        "family": members[name]["family"],
        "oof_auc": roc_auc_score(y, member_oof),
    }
    for fold in range(N_SPLITS):
        mask = fold_ids == fold
        row[f"fold_{fold}_auc"] = roc_auc_score(
            y[mask], member_oof[mask]
        )
    score_rows.append(row)

member_scores = pd.DataFrame(score_rows).sort_values("oof_auc", ascending=False)
display(member_scores)

# Keep every independently trained original member, then fill the remaining
# memory budget with the strongest validated public members. This pragmatic
# cap is recorded because full nested member selection would require more RAM.
original_names = [
    name for name in all_member_names
    if not members[name]["family"].startswith("public_oof")
]
eligible_names = []
for name in original_names + member_scores["member"].tolist():
    if name not in eligible_names:
        eligible_names.append(name)
    if len(eligible_names) >= MAX_STACK_MEMBERS:
        break

member_names = eligible_names
member_oof_matrix = np.column_stack(
    [members[name]["oof"] for name in member_names]
).astype("float32")
member_test_matrix = np.column_stack(
    [members[name]["test"] for name in member_names]
).astype("float32")

rank_oof_matrix = np.column_stack([
    percentile_rank(member_oof_matrix[:, index])
    for index in range(member_oof_matrix.shape[1])
]).astype("float32")
rank_test_matrix = np.column_stack([
    percentile_rank(member_test_matrix[:, index])
    for index in range(member_test_matrix.shape[1])
]).astype("float32")

# Correlation on a deterministic 100k-row sample is accurate enough for
# diversity diagnostics and substantially reduces peak memory for large pools.
correlation_sample = np.linspace(
    0,
    len(train) - 1,
    min(100000, len(train)),
    dtype="int64",
)
correlation_matrix = pd.DataFrame(
    rank_oof_matrix[correlation_sample],
    columns=member_names,
).corr(method="pearson")

plot_names = [
    name for name in member_scores["member"].tolist()
    if name in member_names
][:min(25, len(member_names))]
plot_correlation = correlation_matrix.loc[plot_names, plot_names]
plt.figure(figsize=(12, 10))
plt.imshow(plot_correlation, vmin=0.85, vmax=1.0, cmap="viridis")
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(plot_names)), plot_names, rotation=75, ha="right")
plt.yticks(range(len(plot_names)), plot_names)
plt.title("Top OOF members: sampled rank correlation")
plt.tight_layout()
plt.show()

eligible_indices = np.arange(len(member_names), dtype="int64")

print(f"RAM-safe stacking members: {len(eligible_names)}/{len(all_member_names)}")
for name in eligible_names:
    print(" ", name)

## 18. Fixed V4 rank-logit baseline

This cell reproduces the validated 24-member V4 rank-logit and regime stacks.
It is deliberately retained rather than silently replacing a known baseline.


In [ ]:
# The audit cell already built matrices in eligible order. Reusing references
# avoids four large advanced-indexing copies.
eligible_oof = member_oof_matrix
eligible_test = member_test_matrix
eligible_rank_oof = rank_oof_matrix
eligible_rank_test = rank_test_matrix

def clipped_logit(matrix):
    clipped = np.clip(matrix, 1e-5, 1 - 1e-5)
    return np.log(clipped / (1 - clipped)).astype("float32")

plain_meta_oof = np.column_stack([
    eligible_rank_oof,
    clipped_logit(eligible_oof),
]).astype("float32")
plain_meta_test = np.column_stack([
    eligible_rank_test,
    clipped_logit(eligible_test),
]).astype("float32")

disagreement_oof = eligible_rank_oof.std(axis=1)
disagreement_test = eligible_rank_test.std(axis=1)
disagreement_threshold = float(np.quantile(disagreement_oof, 0.75))

complete_oof = train_features["complete_row"].to_numpy(dtype="float32")
complete_test = test_features["complete_row"].to_numpy(dtype="float32")
many_missing_oof = train_features["many_missing"].to_numpy(dtype="float32")
many_missing_test = test_features["many_missing"].to_numpy(dtype="float32")
high_disagreement_oof = (disagreement_oof >= disagreement_threshold).astype("float32")
high_disagreement_test = (disagreement_test >= disagreement_threshold).astype("float32")

def make_regime_matrix(plain, rank_matrix, complete, many_missing, high_disagreement):
    regimes = [complete, many_missing, high_disagreement]
    interactions = [rank_matrix * regime[:, None] for regime in regimes]
    aggregates = np.column_stack([
        rank_matrix.mean(axis=1),
        rank_matrix.std(axis=1),
        np.median(rank_matrix, axis=1),
        rank_matrix.max(axis=1) - rank_matrix.min(axis=1),
        complete,
        many_missing,
        high_disagreement,
    ])
    return np.column_stack([plain, *interactions, aggregates]).astype("float32")

regime_meta_oof = make_regime_matrix(
    plain_meta_oof,
    eligible_rank_oof,
    complete_oof,
    many_missing_oof,
    high_disagreement_oof,
)
regime_meta_test = make_regime_matrix(
    plain_meta_test,
    eligible_rank_test,
    complete_test,
    many_missing_test,
    high_disagreement_test,
)

def nested_logistic_stack(meta_oof, meta_test, regularization_c):
    stacked_oof = np.zeros(len(train), dtype="float64")
    fold_scores = []

    for fold in range(N_SPLITS):
        train_idx = np.where(fold_ids != fold)[0]
        valid_idx = np.where(fold_ids == fold)[0]
        model = make_pipeline(
            StandardScaler(copy=False),
            LogisticRegression(
                C=regularization_c,
                penalty="l2",
                solver="lbfgs",
                max_iter=600,
                random_state=FOLD_SEED + fold,
            ),
        )
        model.fit(meta_oof[train_idx], y[train_idx])
        stacked_oof[valid_idx] = model.predict_proba(meta_oof[valid_idx])[:, 1]
        fold_scores.append(roc_auc_score(y[valid_idx], stacked_oof[valid_idx]))

    final_model = make_pipeline(
        StandardScaler(copy=False),
        LogisticRegression(
            C=regularization_c,
            penalty="l2",
            solver="lbfgs",
            max_iter=600,
            random_state=FOLD_SEED,
        ),
    )
    final_model.fit(meta_oof, y)
    stacked_test = final_model.predict_proba(meta_test)[:, 1]
    return stacked_oof, stacked_test, fold_scores, final_model

plain_stack_oof, plain_stack_test, plain_fold_scores, plain_model = nested_logistic_stack(
    plain_meta_oof, plain_meta_test, regularization_c=0.10
)
regime_stack_oof, regime_stack_test, regime_fold_scores, regime_model = nested_logistic_stack(
    regime_meta_oof, regime_meta_test, regularization_c=0.03
)

del plain_meta_oof, plain_meta_test, regime_meta_oof, regime_meta_test
del eligible_oof, eligible_test, plain_model, regime_model
gc.collect()

precommitted_mix_oof = (
    (2.0 / 3.0) * percentile_rank(plain_stack_oof)
    + (1.0 / 3.0) * percentile_rank(regime_stack_oof)
)
precommitted_mix_test = (
    (2.0 / 3.0) * percentile_rank(plain_stack_test)
    + (1.0 / 3.0) * percentile_rank(regime_stack_test)
)

rank_average_oof = eligible_rank_oof.mean(axis=1)
rank_average_test = eligible_rank_test.mean(axis=1)

# Build a deliberately diverse three-member rank average.
best_name = member_scores.iloc[0]["member"]
diverse_names = [best_name]
remaining = [name for name in eligible_names if name != best_name]
while remaining and len(diverse_names) < min(3, len(eligible_names)):
    candidate = min(
        remaining,
        key=lambda name: correlation_matrix.loc[name, diverse_names].mean(),
    )
    diverse_names.append(candidate)
    remaining.remove(candidate)

diverse_indices = [member_names.index(name) for name in diverse_names]
diverse_rank_oof = rank_oof_matrix[:, diverse_indices].mean(axis=1)
diverse_rank_test = rank_test_matrix[:, diverse_indices].mean(axis=1)

best_single_index = member_names.index(best_name)
candidates = {
    "best_single": (
        member_oof_matrix[:, best_single_index],
        member_test_matrix[:, best_single_index],
    ),
    "rank_average": (rank_average_oof, rank_average_test),
    "dual_logistic": (plain_stack_oof, plain_stack_test),
    "dual_regime_precommitted": (precommitted_mix_oof, precommitted_mix_test),
    "diverse_three_rank": (diverse_rank_oof, diverse_rank_test),
}

print("Diverse member set:", diverse_names)
print("Plain stack fold AUC:", [f"{score:.6f}" for score in plain_fold_scores])
print("Regime stack fold AUC:", [f"{score:.6f}" for score in regime_fold_scores])

## 19. Build a compact all-member rank matrix

ROC AUC depends only on ordering. Each of the 88 aligned OOF and test streams is
converted once to `float32` percentile ranks. The matrices occupy well under one
gigabyte and avoid the much larger rank, logit, and regime copies that caused the
original full-stack notebook to exhaust Colab RAM.


In [ ]:
greedy_member_names = list(all_member_names)
greedy_member_families = [members[name]["family"] for name in greedy_member_names]
greedy_name_to_index = {
    name: index for index, name in enumerate(greedy_member_names)
}

all_rank_oof = np.empty(
    (len(train), len(greedy_member_names)), dtype="float32"
)
all_rank_test = np.empty(
    (len(test), len(greedy_member_names)), dtype="float32"
)

for index, name in enumerate(greedy_member_names):
    all_rank_oof[:, index] = percentile_rank(members[name]["oof"])
    all_rank_test[:, index] = percentile_rank(members[name]["test"])

member_fold_auc_matrix = np.empty(
    (len(greedy_member_names), N_SPLITS), dtype="float64"
)
for member_index, name in enumerate(greedy_member_names):
    for fold in range(N_SPLITS):
        mask = fold_ids == fold
        member_fold_auc_matrix[member_index, fold] = roc_auc_score(
            y[mask], all_rank_oof[mask, member_index]
        )

rank_memory_mb = (all_rank_oof.nbytes + all_rank_test.nbytes) / (1024 ** 2)
print(f"All-member rank matrices: {all_rank_oof.shape}, {all_rank_test.shape}")
print(f"Rank-matrix memory: {rank_memory_mb:.1f} MB")


## 20. Stability-aware ensemble selection

This is a Caruana-style selector with replacement: selecting a member more than
once increases its final discrete weight. At every step, a cheap gradient and
diversity screen proposes a shortlist. Only the shortlist is evaluated with AUC.

A step is accepted when it improves mean AUC across the active folds, wins on all
but at most one fold, and does not exceed the precommitted worst-fold tolerance.
This makes the search both memory-safe and resistant to a single lucky split.


In [ ]:
def stratified_subsample(indices, labels, max_rows, seed):
    indices = np.asarray(indices, dtype="int64")
    if len(indices) <= max_rows:
        return indices

    rng = np.random.default_rng(seed)
    positive = indices[labels[indices] == 1]
    negative = indices[labels[indices] == 0]
    positive_rows = int(round(max_rows * len(positive) / len(indices)))
    positive_rows = min(max(1, positive_rows), len(positive))
    negative_rows = min(max_rows - positive_rows, len(negative))

    sample = np.concatenate([
        rng.choice(positive, positive_rows, replace=False),
        rng.choice(negative, negative_rows, replace=False),
    ])
    rng.shuffle(sample)
    return sample


def auc_by_fold(labels, prediction, fold_vector, active_folds):
    return np.asarray([
        roc_auc_score(
            labels[fold_vector == fold],
            prediction[fold_vector == fold],
        )
        for fold in active_folds
    ], dtype="float64")


def apply_discrete_ensemble(matrix, unique_indices, weights):
    prediction = np.zeros(matrix.shape[0], dtype="float64")
    for member_index, weight in zip(unique_indices, weights):
        prediction += float(weight) * matrix[:, int(member_index)]
    return prediction


def greedy_ensemble_select(
    rank_matrix,
    labels,
    fold_vector,
    active_folds,
    names,
    families,
    fold_auc_matrix,
    seed,
):
    active_folds = np.asarray(active_folds, dtype="int64")
    selection_indices = np.flatnonzero(np.isin(fold_vector, active_folds))
    search_indices = stratified_subsample(
        selection_indices,
        labels,
        GREEDY_SEARCH_ROWS,
        seed,
    )

    individual_mean_auc = fold_auc_matrix[:, active_folds].mean(axis=1)
    start_index = int(np.argmax(individual_mean_auc))
    current_sum = rank_matrix[:, start_index].astype("float64", copy=True)
    sequence = [start_index]
    current_fold_scores = fold_auc_matrix[start_index, active_folds].copy()
    history = [{
        "step": 1,
        "member": names[start_index],
        "mean_fold_auc": float(current_fold_scores.mean()),
        "mean_gain": np.nan,
        "fold_wins": len(active_folds),
        "minimum_fold_delta": np.nan,
    }]

    required_wins = max(1, len(active_folds) - 1)

    for step in range(2, GREEDY_MAX_STEPS + 1):
        current_prediction = current_sum / (step - 1)

        # A one-dimensional calibration makes the log-loss residual meaningful
        # while keeping candidate screening inexpensive.
        calibration = LogisticRegression(
            C=0.1,
            solver="lbfgs",
            max_iter=120,
            random_state=seed + step,
        )
        calibration.fit(
            current_prediction[search_indices, None],
            labels[search_indices],
        )
        calibrated = calibration.predict_proba(
            current_prediction[selection_indices, None]
        )[:, 1]
        residual = np.zeros(len(labels), dtype="float32")
        residual[selection_indices] = (
            labels[selection_indices] - calibrated
        ).astype("float32")
        gradient = np.asarray(rank_matrix.T @ residual, dtype="float64")

        centered = np.zeros(len(labels), dtype="float32")
        centered_selection = current_prediction[selection_indices]
        centered[selection_indices] = (
            centered_selection - centered_selection.mean()
        ).astype("float32")
        covariance = np.abs(
            np.asarray(rank_matrix.T @ centered, dtype="float64")
        )

        gradient_order = np.argsort(gradient)[::-1][:GREEDY_TOP_GRADIENT]
        individual_order = np.argsort(individual_mean_auc)[::-1]
        top_individual = individual_order[:GREEDY_TOP_INDIVIDUAL]

        strong_pool = individual_order[:min(40, len(individual_order))]
        diverse_order = strong_pool[
            np.argsort(covariance[strong_pool])[:GREEDY_TOP_DIVERSE]
        ]

        family_champions = []
        for family in sorted(set(families)):
            family_indices = np.asarray([
                index for index, value in enumerate(families)
                if value == family
            ], dtype="int64")
            champion = family_indices[
                np.argmax(individual_mean_auc[family_indices])
            ]
            family_champions.append(int(champion))

        shortlist = []
        for member_index in np.concatenate([
            gradient_order,
            top_individual,
            diverse_order,
            np.asarray(family_champions, dtype="int64"),
        ]):
            member_index = int(member_index)
            if member_index not in shortlist:
                shortlist.append(member_index)

        coarse_rows = []
        for member_index in shortlist:
            candidate = (
                current_sum[search_indices]
                + rank_matrix[search_indices, member_index]
            ) / step
            coarse_rows.append((
                roc_auc_score(labels[search_indices], candidate),
                member_index,
            ))

        confirm_indices = [
            member_index
            for _, member_index in sorted(coarse_rows, reverse=True)[
                :GREEDY_CONFIRM_TOP
            ]
        ]

        stable_options = []
        for member_index in confirm_indices:
            candidate_prediction = (
                current_sum + rank_matrix[:, member_index]
            ) / step
            candidate_fold_scores = auc_by_fold(
                labels,
                candidate_prediction,
                fold_vector,
                active_folds,
            )
            fold_deltas = candidate_fold_scores - current_fold_scores
            mean_gain = float(fold_deltas.mean())
            fold_wins = int(np.sum(fold_deltas > 0))
            minimum_fold_delta = float(fold_deltas.min())

            if (
                mean_gain >= GREEDY_MIN_MEAN_GAIN
                and fold_wins >= required_wins
                and minimum_fold_delta >= -GREEDY_MAX_FOLD_LOSS
            ):
                stable_options.append({
                    "member_index": int(member_index),
                    "fold_scores": candidate_fold_scores,
                    "mean_gain": mean_gain,
                    "fold_wins": fold_wins,
                    "minimum_fold_delta": minimum_fold_delta,
                })

        if not stable_options:
            break

        best_option = max(
            stable_options,
            key=lambda item: (
                item["fold_scores"].mean(),
                item["minimum_fold_delta"],
            ),
        )
        selected_index = best_option["member_index"]
        current_sum += rank_matrix[:, selected_index]
        sequence.append(selected_index)
        current_fold_scores = best_option["fold_scores"]
        history.append({
            "step": step,
            "member": names[selected_index],
            "mean_fold_auc": float(current_fold_scores.mean()),
            "mean_gain": best_option["mean_gain"],
            "fold_wins": best_option["fold_wins"],
            "minimum_fold_delta": best_option["minimum_fold_delta"],
        })

    counts = np.bincount(sequence, minlength=rank_matrix.shape[1])
    unique_indices = np.flatnonzero(counts)
    weights = counts[unique_indices].astype("float64") / len(sequence)
    final_prediction = current_sum / len(sequence)

    return {
        "sequence_indices": sequence,
        "sequence_names": [names[index] for index in sequence],
        "unique_indices": unique_indices,
        "unique_names": [names[index] for index in unique_indices],
        "weights": weights,
        "prediction": final_prediction,
        "fold_scores": current_fold_scores,
        "history": history,
    }


## 21. Nested greedy selection and missingness-aware meta-model

For each held-out fold, V5 reruns member selection on the other folds. The selected
blend is then evaluated on untouched rows. A compact logistic model receives the
selected ranks, their logits, ensemble disagreement, and interactions with missingness
regimes. Final test weights are refitted on all OOF rows only after this nested estimate.


In [ ]:
def build_selected_meta_features(
    rank_matrix,
    row_indices,
    selection,
    behavior_frame,
    disagreement_threshold=None,
):
    row_indices = np.asarray(row_indices, dtype="int64")
    selected = np.column_stack([
        rank_matrix[row_indices, int(member_index)]
        for member_index in selection["unique_indices"]
    ]).astype("float32")
    weights = np.asarray(selection["weights"], dtype="float32")
    blend = selected @ weights
    clipped = np.clip(selected, 1e-5, 1 - 1e-5)
    logits = np.log(clipped / (1 - clipped)).astype("float32")
    disagreement = selected.std(axis=1)

    if disagreement_threshold is None:
        disagreement_threshold = float(np.quantile(disagreement, 0.75))

    complete = behavior_frame["complete_row"].to_numpy(dtype="float32")[row_indices]
    many_missing = behavior_frame["many_missing"].to_numpy(dtype="float32")[row_indices]
    missing_fraction = (
        behavior_frame["missing_count"].to_numpy(dtype="float32")[row_indices]
        / len(RAW_COLUMNS)
    )
    high_disagreement = (
        disagreement >= disagreement_threshold
    ).astype("float32")

    aggregates = np.column_stack([
        blend,
        selected.mean(axis=1),
        disagreement,
        selected.max(axis=1) - selected.min(axis=1),
        complete,
        many_missing,
        missing_fraction,
        high_disagreement,
        blend * complete,
        blend * many_missing,
        blend * high_disagreement,
        disagreement * missing_fraction,
    ]).astype("float32")

    features = np.column_stack([
        selected,
        logits,
        aggregates,
    ]).astype("float32")
    return features, disagreement_threshold


greedy_rank_nested_oof = np.zeros(len(train), dtype="float64")
greedy_regime_nested_oof = np.zeros(len(train), dtype="float64")
greedy_rank_test_folds = []
greedy_regime_test_folds = []
greedy_fold_selections = []
greedy_history_rows = []

for heldout_fold in range(N_SPLITS):
    active_folds = [fold for fold in range(N_SPLITS) if fold != heldout_fold]
    train_indices = np.flatnonzero(fold_ids != heldout_fold)
    valid_indices = np.flatnonzero(fold_ids == heldout_fold)

    selection = greedy_ensemble_select(
        all_rank_oof,
        y,
        fold_ids,
        active_folds,
        greedy_member_names,
        greedy_member_families,
        member_fold_auc_matrix,
        seed=FOLD_SEED + 1000 + heldout_fold,
    )
    greedy_fold_selections.append(selection)

    greedy_rank_nested_oof[valid_indices] = selection["prediction"][valid_indices]
    fold_test_prediction = apply_discrete_ensemble(
        all_rank_test,
        selection["unique_indices"],
        selection["weights"],
    )
    greedy_rank_test_folds.append(fold_test_prediction)

    meta_train, threshold = build_selected_meta_features(
        all_rank_oof,
        train_indices,
        selection,
        train_features,
    )
    meta_valid, _ = build_selected_meta_features(
        all_rank_oof,
        valid_indices,
        selection,
        train_features,
        threshold,
    )
    meta_test, _ = build_selected_meta_features(
        all_rank_test,
        np.arange(len(test)),
        selection,
        test_features,
        threshold,
    )

    meta_model = make_pipeline(
        StandardScaler(copy=False),
        LogisticRegression(
            C=GREEDY_META_C,
            penalty="l2",
            solver="lbfgs",
            max_iter=600,
            random_state=FOLD_SEED + 2000 + heldout_fold,
        ),
    )
    meta_model.fit(meta_train, y[train_indices])
    greedy_regime_nested_oof[valid_indices] = meta_model.predict_proba(
        meta_valid
    )[:, 1]
    greedy_regime_test_folds.append(
        meta_model.predict_proba(meta_test)[:, 1]
    )

    for record in selection["history"]:
        greedy_history_rows.append({
            "heldout_fold": heldout_fold,
            **record,
        })

    print(
        f"Held-out fold {heldout_fold}: "
        f"rank AUC={roc_auc_score(y[valid_indices], greedy_rank_nested_oof[valid_indices]):.6f}, "
        f"regime AUC={roc_auc_score(y[valid_indices], greedy_regime_nested_oof[valid_indices]):.6f}"
    )
    print("  sequence:", selection["sequence_names"])

    del meta_train, meta_valid, meta_test, meta_model
    gc.collect()

greedy_rank_foldbag_test = np.mean(greedy_rank_test_folds, axis=0)
greedy_regime_foldbag_test = np.mean(greedy_regime_test_folds, axis=0)

full_greedy_selection = greedy_ensemble_select(
    all_rank_oof,
    y,
    fold_ids,
    list(range(N_SPLITS)),
    greedy_member_names,
    greedy_member_families,
    member_fold_auc_matrix,
    seed=FOLD_SEED + 3000,
)
greedy_rank_full_test = apply_discrete_ensemble(
    all_rank_test,
    full_greedy_selection["unique_indices"],
    full_greedy_selection["weights"],
)

full_meta_train, full_threshold = build_selected_meta_features(
    all_rank_oof,
    np.arange(len(train)),
    full_greedy_selection,
    train_features,
)
full_meta_test, _ = build_selected_meta_features(
    all_rank_test,
    np.arange(len(test)),
    full_greedy_selection,
    test_features,
    full_threshold,
)
full_meta_model = make_pipeline(
    StandardScaler(copy=False),
    LogisticRegression(
        C=GREEDY_META_C,
        penalty="l2",
        solver="lbfgs",
        max_iter=600,
        random_state=FOLD_SEED + 3000,
    ),
)
full_meta_model.fit(full_meta_train, y)
greedy_regime_full_test = full_meta_model.predict_proba(full_meta_test)[:, 1]

candidates["greedy_rank_nested"] = (
    greedy_rank_nested_oof,
    greedy_rank_full_test,
)
candidates["greedy_regime_nested"] = (
    greedy_regime_nested_oof,
    greedy_regime_full_test,
)

print("Full-data greedy sequence:", full_greedy_selection["sequence_names"])
print(
    "Nested greedy rank OOF AUC:",
    f"{roc_auc_score(y, greedy_rank_nested_oof):.6f}",
)
print(
    "Nested greedy regime OOF AUC:",
    f"{roc_auc_score(y, greedy_regime_nested_oof):.6f}",
)

del full_meta_train, full_meta_test, full_meta_model
del all_rank_oof, all_rank_test
gc.collect()


## 22. Stability report and final OOF selection

The primary candidate must improve overall OOF AUC and remain stable across folds.
Nested V5 candidates are compared with the best single member and the fixed V4
baseline. Selection still happens before any new public-leaderboard observation.


In [ ]:
reference_oof = candidates["best_single"][0]
candidate_rows = []

for name, (oof_prediction, _) in candidates.items():
    row = {
        "candidate": name,
        "oof_auc": roc_auc_score(y, oof_prediction),
    }
    fold_deltas = []
    for fold in range(N_SPLITS):
        mask = fold_ids == fold
        fold_auc = roc_auc_score(y[mask], oof_prediction[mask])
        reference_auc = roc_auc_score(y[mask], reference_oof[mask])
        row[f"fold_{fold}_auc"] = fold_auc
        fold_deltas.append(fold_auc - reference_auc)

    fold_deltas = np.asarray(fold_deltas)
    row["fold_wins_vs_best_single"] = int(np.sum(fold_deltas > 0))
    row["minimum_fold_delta"] = float(np.min(fold_deltas))
    row["mean_fold_delta"] = float(np.mean(fold_deltas))
    row["stable_vs_best_single"] = bool(
        name == "best_single"
        or (
            row["fold_wins_vs_best_single"] >= N_SPLITS - 1
            and row["minimum_fold_delta"] >= -GREEDY_MAX_FOLD_LOSS
        )
    )
    candidate_rows.append(row)

candidate_scores = pd.DataFrame(candidate_rows).sort_values(
    "oof_auc", ascending=False
).reset_index(drop=True)
display(candidate_scores)

HONEST_CANDIDATES = [
    "best_single",
    "rank_average",
    "dual_logistic",
    "dual_regime_precommitted",
    "greedy_rank_nested",
    "greedy_regime_nested",
]
honest_table = candidate_scores[
    candidate_scores["candidate"].isin(HONEST_CANDIDATES)
].copy()
stable_table = honest_table[honest_table["stable_vs_best_single"]]
if stable_table.empty:
    stable_table = honest_table

selected_candidate = stable_table.sort_values(
    "oof_auc", ascending=False
).iloc[0]["candidate"]
selected_oof, selected_test = candidates[selected_candidate]

print(f"Selected by nested OOF and fold stability: {selected_candidate}")
print(f"Selected OOF AUC:                       {roc_auc_score(y, selected_oof):.6f}")


## 23. Create validated V5 submissions

The primary file is selected by nested OOF and fold stability. Separate greedy-rank,
greedy-regime, and fold-bagged files are exported for research comparison, but they
should not all be submitted merely to search public-leaderboard weights.


In [ ]:
def build_submission(prediction, filename):
    prediction = np.asarray(prediction, dtype="float64")
    submission = sample_submission[[ID_COL]].copy()
    submission[TARGET] = np.clip(prediction, 0.0, 1.0)

    assert len(submission) == len(test)
    assert submission[ID_COL].equals(sample_submission[ID_COL])
    assert submission[TARGET].notna().all()
    assert np.isfinite(submission[TARGET]).all()
    assert submission[TARGET].between(0, 1).all()
    assert submission[TARGET].nunique() > 1000

    path = SUBMISSION_DIR / filename
    submission.to_csv(path, index=False)
    print("Saved:", path)
    return path

primary_submission_path = build_submission(
    selected_test,
    "submission_s6e8_v5_primary_nested.csv",
)
greedy_rank_submission_path = build_submission(
    greedy_rank_full_test,
    "submission_s6e8_v5_greedy_rank.csv",
)
greedy_regime_submission_path = build_submission(
    greedy_regime_full_test,
    "submission_s6e8_v5_greedy_regime.csv",
)
greedy_foldbag_submission_path = build_submission(
    greedy_regime_foldbag_test,
    "submission_s6e8_v5_greedy_regime_foldbag.csv",
)

display(pd.read_csv(primary_submission_path).head())


## 24. Optional blend with an existing public anchor

This step is deliberately optional because the anchor has no aligned OOF predictions.
You may place your existing `0.97128` submission at:

`/content/drive/MyDrive/s6e8_top3_research_v4/anchors/submission_s6e8_community_blend.csv`

If that file is absent, the notebook attempts to download the credited Apache-2.0
public endpoint by Souvik D. Biswas. It creates one conservative rank blend using
90 percent anchor and 10 percent of the V5 signal selected before leaderboard use.


In [ ]:
ANCHOR_PATH = ANCHOR_DIR / "submission_s6e8_community_blend.csv"
ANCHOR_WEIGHT = 0.90
PUBLIC_ANCHOR_KERNEL = "souvikdbiswas/s6e8-top-20-formula-dual-master-rank-blend"
anchor_submission_path = None
anchor_source = "user_provided" if ANCHOR_PATH.exists() else None

if not ANCHOR_PATH.exists() and KAGGLE_TOKEN_READY:
    public_anchor_dir = ANCHOR_DIR / "souvik_public_endpoint"
    public_anchor_dir.mkdir(parents=True, exist_ok=True)
    public_matches = list(public_anchor_dir.rglob("submission.csv"))

    if not public_matches:
        try:
            subprocess.run(
                [
                    "kaggle", "kernels", "output", PUBLIC_ANCHOR_KERNEL,
                    "-p", str(public_anchor_dir),
                ],
                check=True,
            )
            public_matches = list(public_anchor_dir.rglob("submission.csv"))
        except subprocess.CalledProcessError as exc:
            print("Public anchor download failed; continuing without it:", exc)

    if len(public_matches) == 1:
        shutil.copy2(public_matches[0], ANCHOR_PATH)
        anchor_source = PUBLIC_ANCHOR_KERNEL
        print("Downloaded credited public anchor:", PUBLIC_ANCHOR_KERNEL)
    elif len(public_matches) > 1:
        print("Multiple public submission.csv files found; anchor import skipped.")

if ANCHOR_PATH.exists():
    anchor = pd.read_csv(ANCHOR_PATH)
    assert list(anchor.columns) == [ID_COL, TARGET]
    assert len(anchor) == len(test)
    assert anchor[ID_COL].equals(sample_submission[ID_COL])
    assert anchor[TARGET].notna().all()

    anchor_prediction = anchor[TARGET].to_numpy(dtype="float64")
    anchor_blend = (
        ANCHOR_WEIGHT * percentile_rank(anchor_prediction)
        + (1.0 - ANCHOR_WEIGHT) * percentile_rank(selected_test)
    )
    anchor_submission_path = build_submission(
        anchor_blend,
        "submission_s6e8_v5_anchor90_selected10.csv",
    )
    print("Optional anchor blend created. Treat it as one public experiment only.")
else:
    print("No anchor file found. The original V5 submissions remain fully usable.")
    print("Expected optional path:", ANCHOR_PATH)


## 25. Save the complete V5 research record

OOF candidates, test predictions, member scores, nested selection histories, discrete
full-data weights, provenance, configuration, and runtime information are exported.
These artifacts make the result reproducible and explainable in a technical interview.


In [ ]:
oof_artifact = pd.DataFrame({
    ID_COL: train[ID_COL],
    TARGET: y,
    "fold": fold_ids,
})
test_artifact = pd.DataFrame({ID_COL: test[ID_COL]})

for name, (oof_prediction, test_prediction) in candidates.items():
    oof_artifact[f"candidate__{name}"] = oof_prediction
    test_artifact[f"candidate__{name}"] = test_prediction

oof_artifact.to_csv(ARTIFACT_DIR / "oof_candidate_predictions.csv", index=False)
test_artifact.to_csv(ARTIFACT_DIR / "test_candidate_predictions.csv", index=False)

if SAVE_FULL_MEMBER_MATRIX:
    save_npz_atomic(
        ARTIFACT_DIR / "v4_baseline_member_predictions.npz",
        member_names=np.asarray(member_names),
        oof=member_oof_matrix,
        test=member_test_matrix,
    )

member_scores.to_csv(ARTIFACT_DIR / "member_scores.csv", index=False)
candidate_scores.to_csv(ARTIFACT_DIR / "candidate_scores.csv", index=False)
correlation_matrix.to_csv(ARTIFACT_DIR / "v4_member_rank_correlations.csv")
pd.DataFrame(training_records).to_csv(
    ARTIFACT_DIR / "training_records.csv", index=False
)
pd.DataFrame(greedy_history_rows).to_csv(
    ARTIFACT_DIR / "nested_greedy_history.csv", index=False
)
pd.DataFrame(full_greedy_selection["history"]).to_csv(
    ARTIFACT_DIR / "full_greedy_history.csv", index=False
)
pd.DataFrame({
    "member": full_greedy_selection["unique_names"],
    "weight": full_greedy_selection["weights"],
}).sort_values("weight", ascending=False).to_csv(
    ARTIFACT_DIR / "full_greedy_weights.csv", index=False
)

fold_selection_record = []
for heldout_fold, selection in enumerate(greedy_fold_selections):
    fold_selection_record.append({
        "heldout_fold": heldout_fold,
        "sequence": selection["sequence_names"],
        "members": selection["unique_names"],
        "weights": [float(value) for value in selection["weights"]],
        "selection_fold_auc": [float(value) for value in selection["fold_scores"]],
    })

with open(
    ARTIFACT_DIR / "nested_fold_selections.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(fold_selection_record, file, indent=2)

run_summary = {
    "base_configuration": base_config_for_hash,
    "experiment_configuration": experiment_config_for_hash,
    "base_config_signature": CONFIG_SIGNATURE,
    "experiment_signature": EXPERIMENT_SIGNATURE,
    "gpu": GPU_DESCRIPTION,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "xgboost": xgb.__version__,
    "train_rows": len(train),
    "test_rows": len(test),
    "members": all_member_names,
    "v4_baseline_members": eligible_names,
    "v4_diverse_members": diverse_names,
    "full_greedy_sequence": full_greedy_selection["sequence_names"],
    "full_greedy_members": full_greedy_selection["unique_names"],
    "full_greedy_weights": [
        float(value) for value in full_greedy_selection["weights"]
    ],
    "selected_candidate": selected_candidate,
    "selected_oof_auc": float(roc_auc_score(y, selected_oof)),
    "primary_submission": str(primary_submission_path),
    "anchor_submission": str(anchor_submission_path) if anchor_submission_path else None,
    "anchor_source": anchor_source,
}

with open(ARTIFACT_DIR / "run_summary.json", "w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2)

bundle_root = PERSIST_ROOT / f"s6e8_v5_bundle_{EXPERIMENT_SIGNATURE}"
bundle_zip = Path(shutil.make_archive(
    str(bundle_root),
    "zip",
    root_dir=SUBMISSION_DIR,
))

print("Research artifacts:", ARTIFACT_DIR)
print("Submission files:", SUBMISSION_DIR)
print("Submission bundle:", bundle_zip)


In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(primary_submission_path))

## How to use the V5 results

1. Select a T4 GPU and run the notebook with `FULL_RUN = True`.
2. Confirm that the V4 member cells say `Loaded completed member`; this means the
   expensive base-model checkpoints were reused.
3. Read the final stability table before submitting anything.
4. Submit `submission_s6e8_v5_primary_nested.csv` first.
5. Keep the best previous `0.97128` submission selected until V5 scores higher.
6. Use the optional anchor blend for only one precommitted public experiment.
7. Choose the two final Kaggle entries using OOF evidence and genuine diversity,
   remembering that the private leaderboard contains most of the test rows.

## What V5 improves

- all aligned members can compete without constructing the original full regime matrix;
- member weights are learned through selection with replacement;
- every held-out fold receives a selection made without its labels;
- fold stability is an explicit acceptance condition;
- the missingness-aware meta-model is trained only after member selection;
- every decision and public-member source is exported.

## What this notebook cannot guarantee

The strongest competitors may have private features, models, or generator insights.
A Top-3 position cannot be guaranteed. V5 is designed to maximize defensible validation
quality and private-leaderboard robustness, not to manufacture a promise from a filename.

## Public research that informed the design

- Kodai Fukuda: exact-value target encoding and source-reference features.
- Tamerlan Omralinov: Lookup Transformer and periodic numerical representations.
- Dariush Afshar: rank-logit and regime-aware stacking.
- Szymon Klapinski: aligned OOF libraries, model diversity, and leakage audits.
- Rich Caruana et al.: ensemble selection from libraries of models.

This implementation trains its own base members, records all prediction provenance,
and separates nested validation estimates from final full-OOF weight fitting.
